Session setup: install, mount Drive, set paths, verify everything

the cell i run at the start of every session. installs the libs colab doesn't have (nuscenes-devkit with no deps so numpy stays untouched, small helpers, fvcore and thop for FLOPs), mounts drive, sets PYTHONPATH so test_codet.py works as a separate process, then checks the GPU, imports, test script, checkpoints and test data. prints ALL CHECKS PASSED at the end or the exact failures.

In [1]:
!pip -q install --no-deps nuscenes-devkit
!pip -q install pyquaternion terminaltables cachetools descartes pycocotools fire tqdm
!pip -q install fvcore thop "setuptools<81"

import os, sys, importlib, subprocess
from google.colab import drive

drive.mount("/content/drive")
base = "/content/drive/MyDrive/CS7000_project"
os.environ["PYTHONPATH"] = base + "/coperception"
sys.path.insert(0, base + "/coperception")

problems = []

import torch, numpy
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("numpy", numpy.__version__, "| torch", torch.__version__)
if not torch.cuda.is_available():
    problems.append("no CUDA GPU")

mods = ["scipy", "sklearn", "matplotlib", "shapely", "cv2", "PIL", "nuscenes"]
mods += ["pyquaternion", "terminaltables", "tqdm", "pycocotools"]
mods += ["fvcore", "thop", "coperception.models.det"]
mods += ["coperception.utils.mean_ap", "coperception.utils.mapping"]
mods += ["coperception.utils.convolutional_rnn"]
for mod in mods:
    try:
        importlib.import_module(mod)
    except Exception as e:
        problems.append("import " + mod + ": " + str(e)[:80])

r = subprocess.run(
    ["python", "test_codet.py", "--help"],
    cwd=base + "/coperception/tools/det",
    capture_output=True,
    text=True,
)
if r.returncode != 0:
    problems.append("test_codet.py: " + r.stderr[-300:])

for m in ["lowerbound", "v2v", "when2com"]:
    p = base + "/checkpoints/det/" + m + "/no_rsu/epoch_100.pth"
    if not os.path.exists(p):
        problems.append("missing checkpoint " + m)

if not os.path.isdir(base + "/v2x_sim_20scenes_det/test"):
    problems.append("missing test data")

print("ALL CHECKS PASSED" if not problems else "PROBLEMS:\n- " + "\n- ".join(problems))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
numpy 2.2.6 | torch 2.11.0+cu130
ALL CHECKS PASSED


How test_codet.py builds the models

printing the part of test_codet.py that creates the models and loads the checkpoint, so my profiling code builds all three the same way. this only prints code, nothing runs.

In [2]:
!sed -n 76,175p /content/drive/MyDrive/CS7000_project/coperception/tools/det/test_codet.py

    )
    validation_data_loader = DataLoader(
        validation_dataset, batch_size=1, shuffle=False, num_workers=num_workers
    )
    print("Validation dataset size:", len(validation_dataset))

    if not args.rsu:
        num_agent -= 1

    if flag == "upperbound" or flag.startswith("lowerbound"):
        model = FaFNet(
            config, layer=args.layer, kd_flag=args.kd_flag, num_agent=num_agent
        )
    elif flag.startswith("when2com") or flag.startswith("who2com"):
        # model = PixelwiseWeightedFusionSoftmax(config, layer=args.layer)
        model = When2com(
            config,
            layer=args.layer,
            warp_flag=args.warp_flag,
            num_agent=num_agent,
            compress_level=compress_level,
            only_v2i=only_v2i,
        )
    elif args.com == "disco":
        model = DiscoNet(
            config,
            layer=args.layer,
            kd_flag=args.kd_flag,
            num_agent=num_agent,
            compress_level=compres

Settings, checkpoint loading and the inference step in test_codet.py

printing three parts of test_codet.py: the top (config and variables like compress_level and only_v2i), the argument defaults, and the part after model creation (checkpoint load and the per-frame loop). this gives me the exact inputs to feed each model when i time it. only prints code, nothing runs.

In [3]:
f = "/content/drive/MyDrive/CS7000_project/coperception/tools/det/test_codet.py"
!sed -n 1,75p {f}
!echo "=========== ARGS"
!grep -n "add_argument" -A 3 {f} | grep -i "layer\|compress\|gnn\|num_agent\|only_v2i\|kd_flag\|nepoch\|batch\|num_worker\|box_com\|apply_late\|scene"
!echo "=========== AFTER MODEL"
!sed -n 175,260p {f}

import argparse
import os
from copy import deepcopy

import seaborn as sns
import torch.optim as optim
from torch.utils.data import DataLoader

from coperception.datasets import V2XSimDet
from coperception.configs import Config, ConfigGlobal
from coperception.utils.CoDetModule import *
from coperception.utils.loss import *
from coperception.utils.mean_ap import eval_map
from coperception.models.det import *
from coperception.utils.detection_util import late_fusion
from coperception.utils.data_util import apply_pose_noise


def check_folder(folder_path):
    if not os.path.exists(folder_path):
        os.mkdir(folder_path)
    return folder_path


@torch.no_grad()
def main(args):
    config = Config("train", binary=True, only_det=True)
    config_global = ConfigGlobal("train", binary=True, only_det=True)

    need_log = args.log
    num_workers = args.nworker
    apply_late_fusion = args.apply_late_fusion
    pose_noise = args.pose_noise
    compress_level = args.compress_level
    only

Argument defaults and the model call in test_codet.py

printing the argument definitions (lines 500 to 590) and the end of the per-frame loop (lines 260 to 300). i need the default values to build each model identically, and the exact predict call so my timing code feeds the model the same input. only prints code.

In [4]:
f = "/content/drive/MyDrive/CS7000_project/coperception/tools/det/test_codet.py"
!sed -n 500,590p {f}
!echo "=========== MODEL CALL"
!sed -n 260,300p {f}

        help="The path to the preprocessed sparse BEV training data",
    )
    parser.add_argument("--nepoch", default=100, type=int, help="Number of epochs")
    parser.add_argument("--nworker", default=1, type=int, help="Number of workers")
    parser.add_argument("--lr", default=0.001, type=float, help="Initial learning rate")
    parser.add_argument("--log", action="store_true", help="Whether to log")
    parser.add_argument("--logpath", default="", help="The path to the output log file")
    parser.add_argument(
        "--resume",
        default="",
        type=str,
        help="The path to the saved model that is loaded to resume training",
    )
    parser.add_argument(
        "--resume_teacher",
        default="",
        type=str,
        help="The path to the saved teacher model that is loaded to resume training",
    )
    parser.add_argument(
        "--layer",
        default=3,
        type=int,
        help="Communicate which layer in the single layer com mode",
 

Week 4, step 1: parameters and model size for all 3 models

builds lowerbound, v2v and when2com exactly like test_codet.py does (same settings, 5 agents, same checkpoint load), then counts parameters and computes real weight size (params x 4 bytes in fp32). also splits parameters by top-level module to show where the weight sits. saves the numbers to results/profiling/params_size.json. checkpoint files also hold optimizer state, so they're about 3x bigger than the real model. only the weights would ship to an edge device, so i report weight size, not file size.

In [5]:
import os, sys, json, torch, torch.nn as nn

base = "/content/drive/MyDrive/CS7000_project"
sys.path.insert(0, base + "/coperception")
from coperception.configs import Config
from coperception.models.det import FaFNet, When2com, V2VNet

device = torch.device("cuda")
num_agent = 5  # 6 agents minus the RSU (no_rsu setting)


def build(name):
    config = Config("train", binary=True, only_det=True)
    config.inference = "activated"
    config.flag = name
    config.split = "test"
    if name == "lowerbound":
        return FaFNet(config, layer=3, kd_flag=0, num_agent=num_agent)
    if name == "when2com":
        return When2com(config, layer=3, warp_flag=0, num_agent=num_agent,
                        compress_level=0, only_v2i=0)
    return V2VNet(config, gnn_iter_times=3, layer=3, layer_channel=256,
                  num_agent=num_agent, compress_level=0, only_v2i=0)


results = {}
for name in ["lowerbound", "v2v", "when2com"]:
    try:
        model = nn.DataParallel(build(name)).to(device)
        path = base + "/checkpoints/det/" + name + "/no_rsu/epoch_100.pth"
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        total = sum(p.numel() for p in model.parameters())
        parts = {}
        for child_name, child in model.module.named_children():
            n = sum(p.numel() for p in child.parameters())
            if n > 0:
                parts[child_name] = n
        results[name] = {
            "params": total,
            "weights_mb": round(total * 4 / 1e6, 2),
            "checkpoint_file_mb": round(os.path.getsize(path) / 1e6, 1),
            "epoch": int(ckpt["epoch"]),
            "by_module": parts,
        }
        del model, ckpt
        torch.cuda.empty_cache()
    except Exception as e:
        results[name] = {"error": type(e).__name__ + ": " + str(e)[:200]}

for name, r in results.items():
    print("==", name)
    if "error" in r:
        print("   ERROR", r["error"])
        continue
    print("   params:", format(r["params"], ","),
          "| weights:", r["weights_mb"], "MB (fp32)",
          "| checkpoint file:", r["checkpoint_file_mb"], "MB",
          "| epoch:", r["epoch"])
    for k, v in sorted(r["by_module"].items(), key=lambda x: -x[1]):
        print("     ", k.ljust(24), format(v, ","), str(round(100 * v / r["params"], 1)) + "%")

os.makedirs(base + "/results/profiling", exist_ok=True)
with open(base + "/results/profiling/params_size.json", "w") as f:
    json.dump(results, f, indent=2)
print("saved results/profiling/params_size.json")

Msg size:  32   Key size:  1024
== lowerbound
   params: 7,896,656 | weights: 31.59 MB (fp32) | checkpoint file: 94.9 MB | epoch: 100
      stpn                     7,876,448 99.7%
      regression               10,500 0.1%
      classification           9,708 0.1%
== v2v
   params: 21,083,056 | weights: 84.33 MB (fp32) | checkpoint file: 190.2 MB | epoch: 100
      u_encoder                7,876,448 37.4%
      decoder                  7,876,448 37.4%
      convgru                  5,309,952 25.2%
      regression               10,500 0.0%
      classification           9,708 0.0%
== when2com
   params: 31,296,048 | weights: 125.18 MB (fp32) | checkpoint file: 287.8 MB | epoch: 100
      query_key_net            13,189,472 42.1%
      u_encoder                7,876,448 25.2%
      decoder                  7,876,448 25.2%
      key_net                  1,213,824 3.9%
      query_net                1,085,856 3.5%
      attention_net            33,792 0.1%
      regression               

Next: how the models are called, so I can measure FLOPs, memory and latency

printing the predict_all function from CoDetModule and the forward functions of V2VNet and When2com. i need the exact call signature so my timing code feeds each model real input the same way the test does. it also lets me check if both 7.88M halves are really used. only prints code.

In [6]:
b = "/content/drive/MyDrive/CS7000_project/coperception/coperception"
!grep -n "def predict_all" -A 40 {b}/utils/CoDetModule.py | head -70
!echo "=========== V2VNet forward"
!grep -n "def forward" -A 45 {b}/models/det/V2VNet.py
!echo "=========== When2com forward"
!grep -n "def forward" -A 30 {b}/models/det/When2com.py | head -60

391:    def predict_all(self, data, batch_size, validation=True, num_agent=5):
392-        bev_seq = data["bev_seq"]
393-        # vis_maps = data["vis_maps"]
394-        trans_matrices = data["trans_matrices"]
395-        num_all_agents = data["num_agent"]
396-        # num_sensor = num_all_agents[0, 0]
397-
398-        # if self.MGDA:
399-        #    x = self.encoder(bev_seq)
400-        #    result = self.head(x)
401-        # else:
402-        #    result = self.model(bev_seq, trans_matrices, num_agent_tensor, batch_size=batch_size)
403-        # result = self.model(bev_seq)
404-        if self.config.flag.startswith("when2com") or self.config.flag.startswith(
405-            "who2com"
406-        ):
407-            if self.config.split == "train":
408-                result = self.model(
409-                    bev_seq,
410-                    trans_matrices,
411-                    num_all_agents,
412-                    batch_size=batch_size,
413-                    training=Tr

Helper file: build models, load a real frame, run the forward pass

writes profiling/common.py to drive. it holds the pieces every profiling cell reuses: build and load a model exactly like test_codet.py (5 agents, same settings, bare module in eval mode), load one real test frame through the same dataset pipeline, and run the same forward call the test uses. writing it once keeps the notebook short and means every measurement uses identical setup.

In [10]:
import os
os.makedirs("/content/drive/MyDrive/CS7000_project/profiling", exist_ok=True)
print("ready")

ready


In [11]:
%%writefile /content/drive/MyDrive/CS7000_project/profiling/common.py
import os, sys, itertools, torch, torch.nn as nn
from torch.utils.data import DataLoader

BASE = "/content/drive/MyDrive/CS7000_project"
sys.path.insert(0, BASE + "/coperception")
NUM_AGENT = 5
NAMES = ["lowerbound", "v2v", "when2com"]

from coperception.configs import Config, ConfigGlobal
from coperception.datasets import V2XSimDet
from coperception.models.det import FaFNet, When2com, V2VNet


def _config(name):
    cfg = Config("train", binary=True, only_det=True)
    cfg.inference = "activated"
    cfg.flag = name
    cfg.split = "test"
    return cfg


def load_model(name, device="cuda"):
    cfg = _config(name)
    if name == "lowerbound":
        m = FaFNet(cfg, layer=3, kd_flag=0, num_agent=NUM_AGENT)
    elif name == "when2com":
        m = When2com(cfg, layer=3, warp_flag=0, num_agent=NUM_AGENT,
                     compress_level=0, only_v2i=0)
    else:
        m = V2VNet(cfg, gnn_iter_times=3, layer=3, layer_channel=256,
                   num_agent=NUM_AGENT, compress_level=0, only_v2i=0)
    m = nn.DataParallel(m).to(device)
    path = BASE + "/checkpoints/det/" + name + "/no_rsu/epoch_100.pth"
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    m.load_state_dict(ckpt["model_state_dict"])
    return m.module.eval()


def get_batch(index=0, device="cuda"):
    cfg = _config("lowerbound")
    cfg_g = ConfigGlobal("train", binary=True, only_det=True)
    root = BASE + "/v2x_sim_20scenes_det/test"
    ds = V2XSimDet(
        dataset_roots=[root + "/agent" + str(i) for i in range(1, NUM_AGENT + 1)],
        config=cfg, config_global=cfg_g, split="val", val=True,
        bound="lowerbound", kd_flag=0, rsu=0,
    )
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    sample = next(itertools.islice(iter(loader), index, None))
    (vox, _t, _l, _r, _m, _a, _v, _g, fnames, _tid, nag, trans) = zip(*sample)
    return {
        "bev_seq": torch.cat(tuple(vox), 0).to(device),
        "trans_matrices": torch.stack(tuple(trans), 1).to(device),
        "num_agent": (torch.stack(tuple(nag), 1) - 1).to(device),
        "filenames": fnames[0],
    }


def run(model, name, b):
    with torch.no_grad():
        if name == "when2com":
            return model(b["bev_seq"], b["trans_matrices"], b["num_agent"],
                         batch_size=1, inference="activated", training=False)
        return model(b["bev_seq"], b["trans_matrices"], b["num_agent"], batch_size=1)

Writing /content/drive/MyDrive/CS7000_project/profiling/common.py


Week 4, step 2: peak GPU memory for each model on one real frame

loads one real test frame (5 vehicles at once) and runs each model on it. i do a warm-up run first so cuda setup cost isn't counted, then reset the peak counter and measure. i report weights resident in memory, total peak, and the difference (activations, the temporary feature maps). results save to results/profiling/peak_memory.json. this is pytorch allocated memory only. it excludes the cuda context (about 300 to 400 MB on a T4), so i note that for the nano comparison.

In [12]:
import sys, gc, json, os, torch
sys.path.insert(0, "/content/drive/MyDrive/CS7000_project/profiling")
import common

batch = common.get_batch()
print("frame:", batch["filenames"])
print("bev_seq:", tuple(batch["bev_seq"].shape),
      "| trans:", tuple(batch["trans_matrices"].shape),
      "| num_agent:", batch["num_agent"].tolist())

mem = {}
for name in common.NAMES:
    model = common.load_model(name)
    weights = torch.cuda.memory_allocated() / 1e6
    common.run(model, name, batch)  # warm-up
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    out = common.run(model, name, batch)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / 1e6
    mem[name] = {
        "weights_mb": round(weights, 1),
        "peak_mb": round(peak, 1),
        "activations_mb": round(peak - weights, 1),
    }
    print(name.ljust(11), mem[name])
    del model, out
    gc.collect()
    torch.cuda.empty_cache()

out_dir = "/content/drive/MyDrive/CS7000_project/results/profiling"
os.makedirs(out_dir, exist_ok=True)
with open(out_dir + "/peak_memory.json", "w") as f:
    json.dump({"gpu": torch.cuda.get_device_name(0), "frame": batch["filenames"],
               "results": mem}, f, indent=2)
print("saved peak_memory.json")

The number of val sequences: 300
The number of val sequences: 300
frame: [('/content/drive/MyDrive/CS7000_project/v2x_sim_20scenes_det/test/agent1/5_0/0.npy',)]
bev_seq: (5, 1, 256, 256, 13) | trans: (1, 5, 5, 4, 4) | num_agent: [[4, 4, 4, 4, -1]]
lowerbound  {'weights_mb': 53.9, 'peak_mb': 933.3, 'activations_mb': 879.4}


/content/drive/MyDrive/CS7000_project/coperception/coperception/models/det/base/DetModelBase.py:167: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  grid = F.affine_grid(M, size=torch.Size(size))
/content/drive/MyDrive/CS7000_project/coperception/coperception/models/det/base/DetModelBase.py:168: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warp_feat = F.grid_sample(nb_agent, grid).squeeze()


v2v         {'weights_mb': 107.6, 'peak_mb': 1022.8, 'activations_mb': 915.2}
Msg size:  32   Key size:  1024
activated
activated
when2com    {'weights_mb': 157.5, 'peak_mb': 1069.5, 'activations_mb': 912.0}
saved peak_memory.json


Scan test frames to find ones with all vehicles present

loads the test dataset and checks the active-agent counts on every 15th frame, with the frame file name. i need a fully populated frame for latency, since v2vnet loops over the active agents and a half-empty frame would make it look faster than it is. only reads data, runs no model.

In [13]:
import sys, torch
sys.path.insert(0, "/content/drive/MyDrive/CS7000_project/profiling")
import common
from coperception.configs import ConfigGlobal
from coperception.datasets import V2XSimDet

cfg = common._config("lowerbound")
cfg_g = ConfigGlobal("train", binary=True, only_det=True)
root = common.BASE + "/v2x_sim_20scenes_det/test"
ds = V2XSimDet(
    dataset_roots=[root + "/agent" + str(i) for i in range(1, 6)],
    config=cfg, config_global=cfg_g, split="val", val=True,
    bound="lowerbound", kd_flag=0, rsu=0,
)
print("frames:", len(ds))
for i in range(0, len(ds), 15):
    s = ds[i]
    counts = [int(torch.as_tensor(s[k][10]).flatten()[0]) - 1 for k in range(5)]
    name = str(s[0][8]).split("test/")[-1]
    print(str(i).rjust(4), counts, name)

The number of val sequences: 300
The number of val sequences: 300
frames: 300
   0 [4, 4, 4, 4, -1] agent1/5_0/0.npy']
  15 [4, 4, 4, 4, -1] agent1/5_15/0.npy']
  30 [4, 4, 4, 4, -1] agent1/5_30/0.npy']
  45 [4, 4, 4, 4, -1] agent1/5_45/0.npy']
  60 [4, 4, 4, 4, -1] agent1/5_60/0.npy']
  75 [4, 4, 4, 4, -1] agent1/5_75/0.npy']
  90 [4, 4, 4, 4, -1] agent1/5_90/0.npy']
 105 [5, 5, 5, 5, 5] agent1/8_5/0.npy']
 120 [5, 5, 5, 5, 5] agent1/8_20/0.npy']
 135 [5, 5, 5, 5, 5] agent1/8_35/0.npy']
 150 [5, 5, 5, 5, 5] agent1/8_50/0.npy']
 165 [5, 5, 5, 5, 5] agent1/8_65/0.npy']
 180 [5, 5, 5, 5, 5] agent1/8_80/0.npy']
 195 [5, 5, 5, 5, 5] agent1/8_95/0.npy']
 210 [4, 4, 4, 4, -1] agent1/19_10/0.npy']
 225 [4, 4, 4, 4, -1] agent1/19_25/0.npy']
 240 [4, 4, 4, 4, -1] agent1/19_40/0.npy']
 255 [4, 4, 4, 4, -1] agent1/19_55/0.npy']
 270 [4, 4, 4, 4, -1] agent1/19_70/0.npy']
 285 [4, 4, 4, 4, -1] agent1/19_85/0.npy']


Week 4, step 3: end-to-end latency for all 3 models (5-vehicle and 4-vehicle frames)

times one forward pass for each model: 10 warm-up runs, then 50 timed runs, with a gpu sync before and after each run so the clock is accurate. i do this on a fully populated 5-vehicle frame (scene 8) and a 4-vehicle frame (scene 5) to see how cost scales with vehicles. it measures the model forward only, not box decoding or nms. i report mean, std, median, p95 and save every raw time to results/profiling/latency.json. prints and warnings are muted during the loop so they don't skew the timing.

In [14]:
import sys, gc, json, os, time, io, statistics, warnings, contextlib, torch
sys.path.insert(0, "/content/drive/MyDrive/CS7000_project/profiling")
import common

warnings.filterwarnings("ignore")
WARMUP, RUNS = 10, 50
frames = {"5 vehicles (scene 8)": 150, "4 vehicles (scene 5)": 0}

with contextlib.redirect_stdout(io.StringIO()):
    batches = {k: common.get_batch(index=v) for k, v in frames.items()}


def time_model(model, name, b):
    times = []
    with contextlib.redirect_stdout(io.StringIO()):
        for _ in range(WARMUP):
            common.run(model, name, b)
        for _ in range(RUNS):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            common.run(model, name, b)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    return times


results = {}
for name in common.NAMES:
    with contextlib.redirect_stdout(io.StringIO()):
        model = common.load_model(name)
    results[name] = {}
    for label, b in batches.items():
        t = time_model(model, name, b)
        s = sorted(t)
        results[name][label] = {
            "mean_ms": round(statistics.mean(t), 2),
            "std_ms": round(statistics.stdev(t), 2),
            "median_ms": round(statistics.median(t), 2),
            "p95_ms": round(s[int(0.95 * (len(s) - 1))], 2),
            "min_ms": round(s[0], 2),
            "max_ms": round(s[-1], 2),
            "runs_ms": [round(x, 2) for x in t],
        }
    del model
    gc.collect()
    torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0), "| warmup", WARMUP, "| runs", RUNS)
print("model".ljust(11), "frame".ljust(22), "mean".rjust(8), "std".rjust(7),
      "median".rjust(8), "p95".rjust(8))
for name, d in results.items():
    for label, r in d.items():
        print(name.ljust(11), label.ljust(22), str(r["mean_ms"]).rjust(8),
              str(r["std_ms"]).rjust(7), str(r["median_ms"]).rjust(8),
              str(r["p95_ms"]).rjust(8))

out_dir = "/content/drive/MyDrive/CS7000_project/results/profiling"
os.makedirs(out_dir, exist_ok=True)
with open(out_dir + "/latency.json", "w") as f:
    json.dump({"gpu": torch.cuda.get_device_name(0), "warmup": WARMUP,
               "runs": RUNS, "results": results}, f, indent=2)
print("saved latency.json (times in ms)")

GPU: Tesla T4 | warmup 10 | runs 50
model       frame                      mean     std   median      p95
lowerbound  5 vehicles (scene 8)      97.08    0.51    97.05    97.86
lowerbound  4 vehicles (scene 5)      97.73    0.66    97.79    98.75
v2v         5 vehicles (scene 8)     180.69    3.81   180.62   187.72
v2v         4 vehicles (scene 5)     161.62    2.75   161.05   165.62
when2com    5 vehicles (scene 8)     187.05    1.21   186.93   189.15
when2com    4 vehicles (scene 5)      188.7    0.83    188.9    189.8
saved latency.json (times in ms)


Week 4, step 4: FLOPs per frame, per module, plus estimated Jetson Nano time

counts the real floating-point operations for one forward pass of each model using pytorch's built-in flop counter (it runs the model normally and needs no tracing, so it handles the python loops in v2vnet). i use the 5-vehicle frame and get a total plus a per-module split. then i convert to an estimated nano time as flops divided by 472 gflops, at 100% (ideal), 30% and 20% utilisation. the utilisation values are my assumption and the true figure is unknown, so i report them as a range. it counts convolutions and matrix multiplies only (not the warp or small elementwise ops), so it slightly undercounts. results save to results/profiling/flops.json.

In [15]:
import sys, gc, json, os, io, warnings, contextlib, torch
sys.path.insert(0, "/content/drive/MyDrive/CS7000_project/profiling")
import common
from torch.utils.flop_counter import FlopCounterMode

warnings.filterwarnings("ignore")
NANO_FLOPS = 472e9

with contextlib.redirect_stdout(io.StringIO()):
    batch = common.get_batch(index=150)  # scene 8, all 5 vehicles

results = {}
for name in common.NAMES:
    with contextlib.redirect_stdout(io.StringIO()):
        model = common.load_model(name)
        common.run(model, name, batch)  # warm-up
        with FlopCounterMode(mods=model, display=False) as fc:
            common.run(model, name, batch)
    total = fc.get_total_flops()
    per_module = {}
    for mod_name, ops in fc.get_flop_counts().items():
        if mod_name != "Global":
            per_module[mod_name] = int(sum(ops.values()))
    results[name] = {
        "total_gflops": round(total / 1e9, 2),
        "nano_ms_ideal": round(total / NANO_FLOPS * 1000, 1),
        "nano_ms_at_30pct": round(total / (NANO_FLOPS * 0.30) * 1000, 1),
        "nano_ms_at_20pct": round(total / (NANO_FLOPS * 0.20) * 1000, 1),
        "per_module_gflops": {k: round(v / 1e9, 2) for k, v in per_module.items()},
    }
    del model, fc
    gc.collect()
    torch.cuda.empty_cache()

print("FLOPs for one 5-vehicle frame (counts multiply and add as 2 ops)")
for name, r in results.items():
    print("==", name, "| total", r["total_gflops"], "GFLOPs")
    print("   est. Nano time: ideal", r["nano_ms_ideal"], "ms | at 30%:",
          r["nano_ms_at_30pct"], "ms | at 20%:", r["nano_ms_at_20pct"], "ms")
    top = sorted(r["per_module_gflops"].items(), key=lambda x: -x[1])[:10]
    for k, v in top:
        pct = round(100 * v / r["total_gflops"], 1)
        print("     ", k.ljust(40), str(v).rjust(8), "GFLOPs", str(pct) + "%")

out_dir = "/content/drive/MyDrive/CS7000_project/results/profiling"
os.makedirs(out_dir, exist_ok=True)
with open(out_dir + "/flops.json", "w") as f:
    json.dump({"frame": batch["filenames"], "results": results}, f, indent=2)
print("saved flops.json")

FLOPs for one 5-vehicle frame (counts multiply and add as 2 ops)
== lowerbound | total 155.8 GFLOPs
   est. Nano time: ideal 330.1 ms | at 30%: 1100.3 ms | at 20%: 1650.4 ms
      FaFNet                                      155.8 GFLOPs 100.0%
      FaFNet.stpn                                142.71 GFLOPs 91.6%
      FaFNet.stpn.conv5_1                         18.12 GFLOPs 11.6%
      FaFNet.stpn.conv6_1                         18.12 GFLOPs 11.6%
      FaFNet.stpn.conv7_1                         18.12 GFLOPs 11.6%
      FaFNet.stpn.conv8_1                         18.12 GFLOPs 11.6%
      FaFNet.regression.box_prediction             6.79 GFLOPs 4.4%
      FaFNet.regression                            6.79 GFLOPs 4.4%
      FaFNet.classification                        6.29 GFLOPs 4.0%
      FaFNet.stpn.conv_pre_2                       6.04 GFLOPs 3.9%
== v2v | total 318.87 GFLOPs
   est. Nano time: ideal 675.6 ms | at 30%: 2251.9 ms | at 20%: 3377.9 ms
      V2VNet                        

Week 4 wrap-up: one summary table with efficiency ratios

reads the four result files and builds the final week 4 table in one place: parameters, weight size, peak memory, t4 latency, gflops per vehicle, t4 utilisation, estimated nano time, and map. then it computes the efficiency ratios (map@0.5 gain over lowerbound per extra mb of weights, per extra gflop per vehicle, and per extra ms on the t4). the map values come from our test runs: lowerbound 0.5538/0.4906, v2vnet 0.7529/0.7150, when2com 0.5008/0.4648 (at iou 0.5 and 0.7). saves it as summary.csv for the report.

In [16]:
import json, csv, os

base = "/content/drive/MyDrive/CS7000_project/results/profiling/"
P = json.load(open(base + "params_size.json"))
M = json.load(open(base + "peak_memory.json"))["results"]
L = json.load(open(base + "latency.json"))["results"]
F = json.load(open(base + "flops.json"))["results"]

MAP = {"lowerbound": (0.5538, 0.4906), "v2v": (0.7529, 0.7150), "when2com": (0.5008, 0.4648)}
FRAME = "5 vehicles (scene 8)"
T4_PEAK = 8.1e12
NANO = 472e9

rows = []
for n in ["lowerbound", "v2v", "when2com"]:
    gf = F[n]["total_gflops"]
    lat = L[n][FRAME]["mean_ms"]
    pv = gf / 5
    rows.append({
        "model": n,
        "params_M": round(P[n]["params"] / 1e6, 2),
        "weights_MB": P[n]["weights_mb"],
        "peak_mem_MB": M[n]["peak_mb"],
        "t4_latency_ms": lat,
        "gflops_per_vehicle": round(pv, 1),
        "t4_util_pct": round(100 * gf * 1e9 / (lat / 1000) / T4_PEAK, 1),
        "nano_ms_ideal": round(pv * 1e9 / NANO * 1000),
        "nano_ms_30pct": round(pv * 1e9 / (NANO * 0.30) * 1000),
        "nano_ms_20pct": round(pv * 1e9 / (NANO * 0.20) * 1000),
        "mAP@0.5": MAP[n][0],
        "mAP@0.7": MAP[n][1],
    })

b = rows[0]
for r in rows:
    d_map = r["mAP@0.5"] - b["mAP@0.5"]
    for key, col in [("gain_per_extra_MB", "weights_MB"),
                     ("gain_per_extra_GFLOP", "gflops_per_vehicle"),
                     ("gain_per_extra_ms", "t4_latency_ms")]:
        extra = r[col] - b[col]
        r[key] = round(d_map / extra, 5) if extra > 0 else ""

cols = list(rows[0].keys())
print(" | ".join(cols))
for r in rows:
    print(" | ".join(str(r[c]) for c in cols))

with open(base + "summary.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(rows)
print("saved summary.csv")

model | params_M | weights_MB | peak_mem_MB | t4_latency_ms | gflops_per_vehicle | t4_util_pct | nano_ms_ideal | nano_ms_30pct | nano_ms_20pct | mAP@0.5 | mAP@0.7 | gain_per_extra_MB | gain_per_extra_GFLOP | gain_per_extra_ms
lowerbound | 7.9 | 31.59 | 933.3 | 97.08 | 31.2 | 19.8 | 66 | 220 | 330 | 0.5538 | 0.4906 |  |  | 
v2v | 21.08 | 84.33 | 1022.8 | 180.69 | 63.8 | 21.8 | 135 | 450 | 676 | 0.7529 | 0.715 | 0.00378 | 0.00611 | 0.00238
when2com | 31.3 | 125.18 | 1069.5 | 187.05 | 61.7 | 20.4 | 131 | 436 | 653 | 0.5008 | 0.4648 | -0.00057 | -0.00174 | -0.00059
saved summary.csv
